# c2py23 Performance Timing Demo

This notebook demonstrates:
- Per-function and per-overload timing with `timing: true`
- Reading timing data via `c2py23.perf.read_perf`
- Comparing c2py23 timing with `%%time` cell magic
- Switching tick source between wall-clock and CPU cycle counter

First, build the module:
```bash
cd examples/timing_demo && c2py23 build wsum.c2py
```

In [ ]:
from __future__ import print_function
import ctypes
import os
import sys
import time

sys.path.insert(0, os.path.join(os.getcwd(), 'examples', 'timing_demo'))
import timing_demomod as tmod
from c2py23.perf import read_perf, reset_perf, read_enabled, set_enabled

## Create test data

In [ ]:
N = 1000000
data_double = (ctypes.c_double * N)(*[float(i % 100) for i in range(N)])
data_float = (ctypes.c_float * N)(*[float(i % 100) for i in range(N)])
WEIGHT = 2.0
print("Created arrays of %d elements (double and float)" % N)

## Initial call (double buffer)

First call triggers the double overload.

In [ ]:
%%time
r = tmod.wsum(data_double, WEIGHT)
print("Result: %.1f" % r)
print("Expected sum: %.1f" % (sum(i % 100 for i in range(N)) * WEIGHT))

## Reading timing data

Use `read_perf(func)` to get per-function timing.  Use `read_perf(func, variant='name')`
for per-overload timing (the variant name matches the `name:` field in `c_overloads`).

In [ ]:
perf = read_perf(tmod.wsum)
print("Per-function timing (wsum):")
for k, v in sorted(perf.items()):
    print("  %s: %s" % (k, v))

print()
perf_double = read_perf(tmod.wsum, variant='double')
print("Per-overload timing (wsum/double):")
for k, v in sorted(perf_double.items()):
    print("  %s: %s" % (k, v))

## Call with float buffer

In [ ]:
%%time
r = tmod.wsum(data_float, WEIGHT)
print("Result: %.1f" % r)

In [ ]:
perf_float = read_perf(tmod.wsum, variant='float')
print("Per-overload timing (wsum/float):")
for k, v in sorted(perf_float.items()):
    print("  %s: %s" % (k, v))

## Compare double vs float overload timing

Run many calls to get stable averages.

In [ ]:
reset_perf(tmod.wsum)

NUM_CALLS = 100
for _ in range(NUM_CALLS):
    tmod.wsum(data_double, WEIGHT)
for _ in range(NUM_CALLS):
    tmod.wsum(data_float, WEIGHT)

pd = read_perf(tmod.wsum, variant='double')
pf = read_perf(tmod.wsum, variant='float')

print("Double overload: %d calls, mean C time = %d ns" % (pd['call_count'], pd['c_mean_ns']))
print("Float  overload: %d calls, mean C time = %d ns" % (pf['call_count'], pf['c_mean_ns']))

if pd['c_mean_ns'] > 0 and pf['c_mean_ns'] > 0:
    ratio = float(pd['c_mean_ns']) / float(pf['c_mean_ns'])
    print("Ratio (double/float): %.2fx" % ratio)

## Toggle timing on/off

Timing can be dynamically enabled or disabled per function.

In [ ]:
print("Timing enabled: %s" % bool(read_enabled(tmod.wsum)))

set_enabled(tmod.wsum, 0)
print("Timing after disable: %s" % bool(read_enabled(tmod.wsum)))

reset_perf(tmod.wsum)
for _ in range(10):
    tmod.wsum(data_double, 1.0)

pd_disabled = read_perf(tmod.wsum, variant='double')
print("Calls recorded while disabled: %d (should be 0)" % pd_disabled['call_count'])

set_enabled(tmod.wsum, 1)

## Switch to CPU cycle counter

The tick source defaults to wall-clock (nanosecond resolution).  Switch to
CPU cycle counter for higher precision (via `rdtsc` on x86, `CNTVCT_EL0` on
aarch64, timebase on ppc64).

In [ ]:
try:
    tmod._c2py_set_tick_source("cycle")
    print("Switched to CPU cycle counter.")
    print("Cycle counter frequency: %s Hz" % tmod._c2py_cycle_counter_frequency)
except RuntimeError as e:
    print("Cycle counter not available: %s" % e)
    print("Staying on wall-clock mode.")

In [ ]:
reset_perf(tmod.wsum)

for _ in range(50):
    tmod.wsum(data_double, WEIGHT)

perf_cycle = read_perf(tmod.wsum, variant='double')
print("Cycle counter timing (wsum/double):")
for k, v in sorted(perf_cycle.items()):
    print("  %s: %s" % (k, v))

if perf_cycle['call_count'] > 0:
    print()
    print("C function took %d cycles = %d ns (average)" % (
        perf_cycle['c_total_cycles'],
        perf_cycle['c_mean_ns']))

## Switch back to wall-clock

In [ ]:
tmod._c2py_set_tick_source("clock")
print("Switched back to wall-clock.")